In [2]:
import os
print(f"Current Working Directory: {os.getcwd()}")

Current Working Directory: /content


In [5]:
from google.colab import drive
drive.mount('/content/drive')

KeyboardInterrupt: 

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import pymc as pm
import arviz as az
import pytensor.tensor as pt
from pathlib import Path
from scipy.stats.mstats import winsorize
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
import warnings

# Suppress PyMC/Theano distinct warnings for cleaner output
warnings.filterwarnings('ignore')

# ===================================================================
# CONFIGURATION
# ===================================================================
DATA_DIR = Path("Data/Verified")
OUTPUT_DIR = Path("Results/Bayesian_TVP")
PLOT_DIR = Path("Plots/Bayesian_TVP")
START_DATE = '2020-01-01'
END_DATE = '2024-01-01'
CRYPTO_VOL = "Delta_LogGK"
STABLE_VOL = "Delta_LogGK"

# BAYESIAN SETTINGS
LAGS = 2                # Keep lags low (1 or 2) for TVP-VAR to avoid overfitting
DRAWS = 200            # Number of MCMC samples (Higher = Better but slower)
TUNE = 100             # Burn-in period
CHAINS = 2              # Number of parallel chains
TARGET_SAMPLE_RATE = 1  # 1 = use all data. Increase (e.g. 5) to downsample for testing speed.
WINSOR_QUANTILE = 0.01 
MIN_PCA_WINDOW = 60 

# ===================================================================
# Helper Functions (Data Loading & PCA - Unchanged)
# ===================================================================

def get_expanding_pca(df, min_periods=30):
    n_samples, n_features = df.shape
    pc_series = np.full(n_samples, np.nan)
    prev_components = None
    
    for t in range(min_periods, n_samples):
        window_data = df.iloc[:t+1].values
        scaler = StandardScaler()
        scaled_window = scaler.fit_transform(window_data)
        
        pca = PCA(n_components=1)
        pca.fit(scaled_window)
        current_components = pca.components_[0]
        
        sign_multiplier = 1.0
        if prev_components is not None:
            if np.dot(prev_components, current_components) < 0:
                sign_multiplier = -1.0
        
        current_obs_scaled = scaler.transform(window_data[-1].reshape(1, -1))
        pc_value = pca.transform(current_obs_scaled)[0, 0]
        pc_series[t] = pc_value * sign_multiplier
        prev_components = current_components * sign_multiplier

    return pd.Series(pc_series, index=df.index)

def load_and_process_factors(data_dir, start_date, end_date, winsor_limit):
    print("Loading and processing data...")
    coin_data = {}
    if not data_dir.exists(): raise FileNotFoundError(f"Data directory not found: {data_dir}")

    for file in data_dir.glob("*.csv"):
        if (c := file.stem.replace("Verif_", "")) in ["DAI", "USDC", "USDT", "BNB", "BTC", "ETH", "XRP"]:
            df = pd.read_csv(file, parse_dates=['Date']).sort_values("Date")
            coin_data[c] = df[(df['Date'] >= start_date) & (df['Date'] <= end_date)].set_index('Date')

    def get_fac(coins, var, name):
        df_list = [coin_data[c][var] for c in coins if c in coin_data]
        if not df_list: return pd.Series(dtype=float, name=name)
        df = pd.concat(df_list, axis=1, keys=coins, join='inner').dropna()
        for col in df.columns: 
            df[col] = winsorize(df[col], limits=[winsor_limit, winsor_limit])
        pca = get_expanding_pca(df, min_periods=MIN_PCA_WINDOW)
        pca.name = name
        return pca

    factors = {
        "Stable_Volume": get_fac(["DAI", "USDC", "USDT"], "LogVolChange", "Stable_Volume"),
        "Stable_Volatility": get_fac(["DAI", "USDC", "USDT"], STABLE_VOL, "Stable_Volatility"),
        "Crypto_Volatility": get_fac(["BNB", "BTC", "ETH", "XRP"], CRYPTO_VOL, "Crypto_Volatility"),
        # Reduced set for Bayesian Performance (6 vars is too heavy for laptop MCMC)
        # We focus on the core Volatility pair for the demo
    }
    return factors

# ===================================================================
# BAYESIAN TVP-VAR IMPLEMENTATION
# ===================================================================

def prepare_lagged_data(df, lags):
    """Creates the matrix X of lagged predictors."""
    n_vars = df.shape[1]
    X_list = []
    
    # Create lags
    for i in range(1, lags + 1):
        shifted = df.shift(i)
        shifted.columns = [f"{col}_L{i}" for col in df.columns]
        X_list.append(shifted)
        
    X = pd.concat(X_list, axis=1).dropna()
    # Align Y to X
    Y = df.loc[X.index]
    
    return Y, X

def run_bayesian_tvp_model(target_name, target_data, predictor_data):
    """
    Fits a Bayesian Time-Varying Parameter Regression with Stochastic Volatility.
    Equation: y_t = X_t * beta_t + exp(h_t/2) * epsilon_t
    """
    print(f"  > Sampling TVP Model for Target: {target_name}...")
    
    # Data prep for PyMC
    # Standardize inputs to help MCMC convergence
    scaler_X = StandardScaler()
    X_scaled = scaler_X.fit_transform(predictor_data)
    
    scaler_y = StandardScaler()
    y_scaled = scaler_y.fit_transform(target_data.values.reshape(-1, 1)).flatten()
    
    n_obs, n_predictors = X_scaled.shape
    
    coords = {
        "time": target_data.index,
        "predictors": predictor_data.columns
    }

    with pm.Model(coords=coords) as model:
        # 1. Stochastic Volatility (SV)
        # modeLog-volatility follows a Random Walk
        sigma_step = pm.Exponential("sigma_step", 1.0) # Step size of volatility drift
        log_vol = pm.GaussianRandomWalk("log_vol", sigma=sigma_step, dims="time", init_dist=pm.Normal.dist(0, 1))
        
        # 2. Time-Varying Coefficients (TVP)
        # Coefficients follow a Random Walk
        # We need a matrix of shape (n_obs, n_predictors)
        
        # Priors for the drift of coefficients
        beta_step = pm.Exponential("beta_step", 0.1) 
        
        # We model independent RWs for each predictor's coefficient
        betas = pm.GaussianRandomWalk(
            "betas", 
            sigma=beta_step, 
            dims=("time", "predictors"),
            init_dist=pm.Normal.dist(0, 1)
        )
        
        # 3. The Linear Model
        # y_est = sum(X * beta, axis=1)
        # Using pytensor dot product for time-varying weights
        # X is (T, K), betas is (T, K) -> Elementwise multiplication then sum
        mu = (X_scaled * betas).sum(axis=1)
        
        # 4. Likelihood
        # The variance is exp(log_vol)
        sigma_t = pm.math.exp(log_vol)
        obs = pm.Normal("obs", mu=mu, sigma=sigma_t, observed=y_scaled)
        
        # 5. Sampling
        trace = pm.sample(draws=DRAWS, tune=TUNE, chains=CHAINS, target_accept=0.9, progressbar=True)
        
    return trace, scaler_X, scaler_y

def extract_and_plot_causality(trace, source_name, target_name, dates, scaler_y, output_dir):
    """
    Extracts posterior, PLOTS it, and SAVES numerical results to CSV.
    """
    # 1. Extract Posteriors (Same as before)
    posterior_betas = trace.posterior["betas"]
    source_lags = [c for c in posterior_betas.coords["predictors"].values if source_name in str(c)]
    
    if not source_lags:
        print(f"Warning: No lags found for {source_name}")
        return

    # Sum coefficients across lags for total impact
    total_impact = posterior_betas.sel(predictors=source_lags).sum(dim="predictors")
    
    # Calculate HDI (94%) and Mean
    hdi = az.hdi(total_impact, hdi_prob=0.94)
    mean_traj = total_impact.mean(dim=["chain", "draw"]).values
    lower_bound = hdi["betas"].sel(hdi="lower").values
    upper_bound = hdi["betas"].sel(hdi="higher").values
    
    # =========================================================
    # NEW: Save Numerical Results to CSV
    # =========================================================
    results_df = pd.DataFrame({
        'Date': dates,
        'Source': source_name,
        'Target': target_name,
        'Beta_Mean': mean_traj,
        'Beta_Lower_94': lower_bound,
        'Beta_Upper_94': upper_bound
    })
    
    # Flag significant days (where 0 is not in the interval)
    # If lower > 0 OR upper < 0, it is significant
    results_df['Significant'] = (results_df['Beta_Lower_94'] > 0) | (results_df['Beta_Upper_94'] < 0)
    
    csv_filename = f"Data_{source_name}_to_{target_name}.csv"
    results_df.to_csv(output_dir / csv_filename, index=False)
    print(f"    -> CSV Saved: {csv_filename}")

    # =========================================================
    # Plotting (Standard)
    # =========================================================
    fig, ax = plt.subplots(figsize=(12, 6))
    
    ax.plot(dates, mean_traj, color="#1f77b4", label="Posterior Mean")
    ax.fill_between(dates, lower_bound, upper_bound, color="#1f77b4", alpha=0.2, label="94% HDI")
    ax.axhline(0, color='red', linestyle='--', linewidth=1)
    
    # Highlight significant periods visually
    sig_dates = results_df[results_df['Significant']]['Date']
    sig_values = results_df[results_df['Significant']]['Beta_Mean']
    if not sig_dates.empty:
        ax.scatter(sig_dates, sig_values, s=5, c='red', alpha=0.6, label='Significant', zorder=3)

    ax.set_title(f"TVP-VAR Coefficient: {source_name} $\\to$ {target_name}")
    ax.set_ylabel("Impact Magnitude")
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.legend()
    plt.tight_layout()
    
    plot_filename = f"Plot_{source_name}_to_{target_name}.png"
    plt.savefig(output_dir / plot_filename)
    plt.close(fig)

# ===================================================================
# Main Execution
# ===================================================================
if __name__ == "__main__":
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    PLOT_DIR.mkdir(parents=True, exist_ok=True)
    
    # 1. Load Data
    # Limiting factors to just Volatility for this heavy computational demo
    all_factors = load_and_process_factors(DATA_DIR, START_DATE, END_DATE, WINSOR_QUANTILE)
    
    # Filter to just the key variables for the Bayesian run
    # (Running MCMC on 6 vars x 5 lags is too slow for a script)
    # We will test: Does Crypto Volatility cause Stable Volatility?
    df = pd.concat([all_factors['Crypto_Volatility'], all_factors['Stable_Volatility']], axis=1).dropna()
    
    # Downsample for testing if needed (Every Nth day)
    if TARGET_SAMPLE_RATE > 1:
        df = df.iloc[::TARGET_SAMPLE_RATE]
        print(f"Downsampled data to {len(df)} observations for speed.")

    # 2. Prepare Lags
    Y, X = prepare_lagged_data(df, LAGS)
    dates = Y.index

    # 3. Run Model for Stable_Volatility (Target)
    # We want to see if Crypto_Volatility (Predictor) affects it
    target = "Stable_Volatility"
    trace, _, _ = run_bayesian_tvp_model(target, Y[target], X)
    
    # 4. Plot Result
    # Source is Crypto_Volatility
    extract_and_plot_causality(trace, "Crypto_Volatility", target, dates, None, PLOT_DIR)
    
    # 5. Run Model for Crypto_Volatility (Target) - Reverse causality
    target_rev = "Crypto_Volatility"
    trace_rev, _, _ = run_bayesian_tvp_model(target_rev, Y[target_rev], X)
    extract_and_plot_causality(trace_rev, "Stable_Volatility", target_rev, dates, None, PLOT_DIR)

    print("-" * 90)
    print(f"Bayesian Analysis Complete.")
    print(f"Plots saved to: {PLOT_DIR.resolve()}")

Loading and processing data...


FileNotFoundError: Data directory not found: Data/Verified